In [1]:
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata
import re

DATA_DIR = "../data"
# Define the column names
NEW_COLUMNS = [
    "Périmètre", "Nature", "Date", "Heures", "Consommation", "Prévision_J_1", "Prévision_J",
    "Fioul", "Charbon", "Gaz", "Nucléaire", "Eolien", "Solaire", "Hydraulique", "Pompage",
    "Bioénergies", "Ech_physiques", "Taux_de_Co2", "Ech_comm_Angleterre", "Ech_comm_Espagne",
    "Ech_comm_Italie", "Ech_comm_Suisse", "Ech_comm_Allemagne_Belgique", "Fioul_TAC",
    "Fioul_Cogén", "Fioul_Autres", "Gaz_TAC", "Gaz_Cogén", "Gaz_CCG", "Gaz_Autres",
    "Hydraulique_Fil_de_leau_Eclusée", "Hydraulique_Lacs", "Hydraulique_STEP_turbinage",
    "Bioénergies_Déchets", "Bioénergies_Biomasse", "Bioénergies_Biogaz", "Stockage_batterie",
    "Déstockage_batterie", "Eolien_terrestre", "Eolien_offshore"
]
SUPPORTED_EXTENSIONS = [".csv", ".xls", ".xlsx"]

In [3]:
def _clean_colname(col: str) -> str:
    if not isinstance(col, str):
        col = str(col)

    col = col.strip()
    col = col.replace("�", "e").replace("?", "e")

    col = unicodedata.normalize("NFKD", col)
    col = col.encode("ascii", "ignore").decode("ascii")

    col = col.lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")

    return col

In [4]:
def _load_single_file(filepath: Path) -> pd.DataFrame:
    suffix = filepath.suffix.lower()

    def read_csv_smart(path):
        best_df = None
        best_cols = 0

        for sep in [";", ",", "\t"]:
            try:
                df_try = pd.read_csv(
                    path,
                    sep=sep,
                    encoding="latin1",
                    low_memory=False,
                    dtype=str,
                    index_col=False
                )                
                if df_try.shape[1] > best_cols:
                    best_cols = df_try.shape[1]
                    best_df = df_try
            except Exception:
                continue

        if best_df is None or best_cols == 1:
            raise ValueError("Impossible de détecter le séparateur CSV")

        return best_df

    try:
        if suffix == ".xls":
                df = read_csv_smart(filepath)
    except Exception as e:
        raise RuntimeError(f"Erreur lors du chargement de {filepath.name} : {e}")

    # Nettoyage des colonnes
    df.columns = [_clean_colname(c) for c in df.columns]
    
    df["source_file"] = filepath.name

    return df

In [5]:
def load_all_data(data_dir: str) -> pd.DataFrame:
    """
    Charge automatiquement tous les fichiers du dossier data/
    """
    data_path = Path(data_dir)

    if not data_path.exists():
        raise FileNotFoundError(f"Dossier introuvable : {data_dir}")

    all_dfs = []

    for file in data_path.iterdir():
        if file.suffix.lower() in SUPPORTED_EXTENSIONS:
            print(f"Chargement : {file.name}")
            df = _load_single_file(file)
            all_dfs.append(df)

    if not all_dfs:
        return pd.DataFrame()

    df_final = pd.concat(all_dfs, ignore_index=True)
    return df_final

In [6]:
df = load_all_data(DATA_DIR)

Chargement : eCO2mix_RTE_Annuel-Definitif_2013.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2014.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2023.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2017.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2016.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2022.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2018.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2021.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2020.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2012.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2015.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2019.xls


In [7]:
# Handle column count mismatch
if len(df.columns) < len(NEW_COLUMNS):
    print(f"Column count mismatch: File has {len(df.columns)} columns, but {len(NEW_COLUMNS)} are expected.")
    print("Adding missing columns with None values...")
    for missing_col in NEW_COLUMNS[len(df.columns):]:
        df[missing_col] = None
elif len(df.columns) > len(NEW_COLUMNS):
    print(f"Column count mismatch: File has {len(df.columns)} columns, but {len(NEW_COLUMNS)} are expected.")
    print("Truncating extra columns...")
    df = df.iloc[:, :len(NEW_COLUMNS)]

# Clean column names
df.columns = [
    col.strip().replace(" ", "_").replace("-", "_").replace("?", "e").lower()
    for col in NEW_COLUMNS
]
print(f"Cleaned column names: {df.columns.tolist()}")

Column count mismatch: File has 41 columns, but 40 are expected.
Truncating extra columns...
Cleaned column names: ['périmètre', 'nature', 'date', 'heures', 'consommation', 'prévision_j_1', 'prévision_j', 'fioul', 'charbon', 'gaz', 'nucléaire', 'eolien', 'solaire', 'hydraulique', 'pompage', 'bioénergies', 'ech_physiques', 'taux_de_co2', 'ech_comm_angleterre', 'ech_comm_espagne', 'ech_comm_italie', 'ech_comm_suisse', 'ech_comm_allemagne_belgique', 'fioul_tac', 'fioul_cogén', 'fioul_autres', 'gaz_tac', 'gaz_cogén', 'gaz_ccg', 'gaz_autres', 'hydraulique_fil_de_leau_eclusée', 'hydraulique_lacs', 'hydraulique_step_turbinage', 'bioénergies_déchets', 'bioénergies_biomasse', 'bioénergies_biogaz', 'stockage_batterie', 'déstockage_batterie', 'eolien_terrestre', 'eolien_offshore']


In [8]:
# Analyze the data types of the DataFrame
print("Analyzing data types...")
data_types = {}
for col in df.columns:
    try:
        # Try to convert the column to numeric
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if df[col].isnull().all():
            data_types[col] = "TEXT"  # If all values are NaN, treat as TEXT
        elif df[col].dropna().apply(float.is_integer).all():
            data_types[col] = "INTEGER"
        else:
            data_types[col] = "FLOAT"
    except Exception:
        data_types[col] = "TEXT"  # Default to TEXT if conversion fails

print("Suggested PostgreSQL data types:")
for col, dtype in data_types.items():
    print(f"{col}: {dtype}")

Analyzing data types...
Suggested PostgreSQL data types:
périmètre: TEXT
nature: TEXT
date: TEXT
heures: TEXT
consommation: INTEGER
prévision_j_1: INTEGER
prévision_j: INTEGER
fioul: INTEGER
charbon: INTEGER
gaz: INTEGER
nucléaire: INTEGER
eolien: INTEGER
solaire: INTEGER
hydraulique: INTEGER
pompage: INTEGER
bioénergies: INTEGER
ech_physiques: INTEGER
taux_de_co2: INTEGER
ech_comm_angleterre: INTEGER
ech_comm_espagne: INTEGER
ech_comm_italie: INTEGER
ech_comm_suisse: INTEGER
ech_comm_allemagne_belgique: INTEGER
fioul_tac: INTEGER
fioul_cogén: INTEGER
fioul_autres: INTEGER
gaz_tac: INTEGER
gaz_cogén: INTEGER
gaz_ccg: INTEGER
gaz_autres: INTEGER
hydraulique_fil_de_leau_eclusée: INTEGER
hydraulique_lacs: INTEGER
hydraulique_step_turbinage: INTEGER
bioénergies_déchets: INTEGER
bioénergies_biomasse: INTEGER
bioénergies_biogaz: INTEGER
stockage_batterie: TEXT
déstockage_batterie: INTEGER
eolien_terrestre: INTEGER
eolien_offshore: INTEGER


In [9]:
# Handle column count mismatch
if len(df.columns) < len(NEW_COLUMNS):
    print(f"Column count mismatch: File has {len(df.columns)} columns, but {len(NEW_COLUMNS)} are expected.")
    print("Adding missing columns with None values...")
    for missing_col in NEW_COLUMNS[len(df.columns):]:
        df[missing_col] = None
elif len(df.columns) > len(NEW_COLUMNS):
    print(f"Column count mismatch: File has {len(df.columns)} columns, but {len(NEW_COLUMNS)} are expected.")
    print("Truncating extra columns...")
    df = df.iloc[:, :len(NEW_COLUMNS)]

# Ensure the number of columns matches the number of new column names
if len(df.columns) != len(NEW_COLUMNS):
    raise ValueError(f"Final column count mismatch: DataFrame has {len(df.columns)} columns, but {len(NEW_COLUMNS)} are expected.")

# Clean column names
df.columns = [
    col.strip().replace(" ", "_").replace("-", "_").replace("?", "e").lower()
    for col in NEW_COLUMNS
]
print(f"Cleaned column names: {df.columns.tolist()}")

Cleaned column names: ['périmètre', 'nature', 'date', 'heures', 'consommation', 'prévision_j_1', 'prévision_j', 'fioul', 'charbon', 'gaz', 'nucléaire', 'eolien', 'solaire', 'hydraulique', 'pompage', 'bioénergies', 'ech_physiques', 'taux_de_co2', 'ech_comm_angleterre', 'ech_comm_espagne', 'ech_comm_italie', 'ech_comm_suisse', 'ech_comm_allemagne_belgique', 'fioul_tac', 'fioul_cogén', 'fioul_autres', 'gaz_tac', 'gaz_cogén', 'gaz_ccg', 'gaz_autres', 'hydraulique_fil_de_leau_eclusée', 'hydraulique_lacs', 'hydraulique_step_turbinage', 'bioénergies_déchets', 'bioénergies_biomasse', 'bioénergies_biogaz', 'stockage_batterie', 'déstockage_batterie', 'eolien_terrestre', 'eolien_offshore']


In [10]:

# Import necessary libraries
import psycopg2
import pandas as pd

# Database credentials
HOST = "localhost"
DB = "postgres"
USER = "postgres"
PASSWORD = "postgres"
PORT = 5441

# Define the table name
table_name = "meteo_all_cities"

# Function to create the table with the correct data types
def create_table(conn, table_name):
    curs = conn.cursor()
    curs.execute(f'''
        CREATE TABLE IF NOT EXISTS {table_name} (
            périmètre TEXT,
            nature TEXT,
            date TEXT,
            heures TEXT,
            consommation INTEGER,
            prévision_j_1 INTEGER,
            prévision_j INTEGER,
            fioul INTEGER,
            charbon INTEGER,
            gaz INTEGER,
            nucléaire INTEGER,
            eolien INTEGER,
            solaire INTEGER,
            hydraulique INTEGER,
            pompage INTEGER,
            bioénergies INTEGER,
            ech_physiques INTEGER,
            taux_de_co2 INTEGER,
            ech_comm_angleterre INTEGER,
            ech_comm_espagne INTEGER,
            ech_comm_italie INTEGER,
            ech_comm_suisse INTEGER,
            ech_comm_allemagne_belgique INTEGER,
            fioul_tac INTEGER,
            fioul_cogén INTEGER,
            fioul_autres INTEGER,
            gaz_tac INTEGER,
            gaz_cogén INTEGER,
            gaz_ccg INTEGER,
            gaz_autres INTEGER,
            hydraulique_fil_de_leau_eclusée INTEGER,
            hydraulique_lacs INTEGER,
            hydraulique_step_turbinage INTEGER,
            bioénergies_déchets INTEGER,
            bioénergies_biomasse INTEGER,
            bioénergies_biogaz INTEGER,
            stockage_batterie TEXT,
            déstockage_batterie INTEGER,
            eolien_terrestre INTEGER,
            eolien_offshore INTEGER
        )
    ''')
    conn.commit()

# Function to cast DataFrame columns to the correct data types
def cast_dataframe_columns(df):
    data_types = {
        "périmètre": "str",
        "nature": "str",
        "date": "str",
        "heures": "str",
        "consommation": "Int64",
        "prévision_j_1": "Int64",
        "prévision_j": "Int64",
        "fioul": "Int64",
        "charbon": "Int64",
        "gaz": "Int64",
        "nucléaire": "Int64",
        "eolien": "Int64",
        "solaire": "Int64",
        "hydraulique": "Int64",
        "pompage": "Int64",
        "bioénergies": "Int64",
        "ech_physiques": "Int64",
        "taux_de_co2": "Int64",
        "ech_comm_angleterre": "Int64",
        "ech_comm_espagne": "Int64",
        "ech_comm_italie": "Int64",
        "ech_comm_suisse": "Int64",
        "ech_comm_allemagne_belgique": "Int64",
        "fioul_tac": "Int64",
        "fioul_cogén": "Int64",
        "fioul_autres": "Int64",
        "gaz_tac": "Int64",
        "gaz_cogén": "Int64",
        "gaz_ccg": "Int64",
        "gaz_autres": "Int64",
        "hydraulique_fil_de_leau_eclusée": "Int64",
        "hydraulique_lacs": "Int64",
        "hydraulique_step_turbinage": "Int64",
        "bioénergies_déchets": "Int64",
        "bioénergies_biomasse": "Int64",
        "bioénergies_biogaz": "Int64",
        "stockage_batterie": "str",
        "déstockage_batterie": "Int64",
        "eolien_terrestre": "Int64",
        "eolien_offshore": "Int64"
    }
    for col, dtype in data_types.items():
        if dtype == "str":
            df[col] = df[col].astype(str)
        elif dtype == "Int64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df

# Function to insert data into the table
def insert_data_from_df(conn, table_name, df):
    curs = conn.cursor()
    for index, row in df.iterrows():
        # Generate a unique ID for each row
        db_id = f"{row['périmètre']}_{row['date']}_{row['heures']}"
        
        # Check if the data already exists
        curs.execute(f"SELECT * FROM {table_name} WHERE périmètre=%s AND date=%s AND heures=%s", 
                     (row['périmètre'], row['date'], row['heures']))
        existing_data = curs.fetchone()

        if not existing_data:
            # Insert the row into the database
            curs.execute(f'''
                INSERT INTO {table_name} (
                    périmètre, nature, date, heures, consommation, prévision_j_1, prévision_j, fioul, charbon, gaz, nucléaire, eolien, solaire, hydraulique, pompage, bioénergies, ech_physiques, taux_de_co2, ech_comm_angleterre, ech_comm_espagne, ech_comm_italie, ech_comm_suisse, ech_comm_allemagne_belgique, fioul_tac, fioul_cogén, fioul_autres, gaz_tac, gaz_cogén, gaz_ccg, gaz_autres, hydraulique_fil_de_leau_eclusée, hydraulique_lacs, hydraulique_step_turbinage, bioénergies_déchets, bioénergies_biomasse, bioénergies_biogaz, stockage_batterie, déstockage_batterie, eolien_terrestre, eolien_offshore
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ''', tuple(row))
            conn.commit()
            print(f"Inserted data: {db_id} into {table_name}")
        else:
            print(f"Data already exists: {db_id} in {table_name}")

# Load the DataFrame (assuming df is already loaded)
df = load_all_data(DATA_DIR)

# Clean column names
df.columns = [
    col.strip().replace(" ", "_").replace("-", "_").replace("?", "e").lower()
    for col in NEW_COLUMNS
]

# Cast DataFrame columns to the correct data types
df = cast_dataframe_columns(df)

# Connect to the PostgreSQL database
conn = psycopg2.connect(database=DB, user=USER, password=PASSWORD, host=HOST, port=PORT)

# Create the table if it doesn't exist
create_table(conn, table_name)

# Insert data into the table
insert_data_from_df(conn, table_name, df)

# Close the connection
conn.close()

Chargement : eCO2mix_RTE_Annuel-Definitif_2013.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2014.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2023.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2017.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2016.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2022.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2018.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2021.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2020.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2012.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2015.xls
Chargement : eCO2mix_RTE_Annuel-Definitif_2019.xls


ValueError: Length mismatch: Expected axis has 41 elements, new values have 40 elements